# Contrastive/Cosine similarity loss finetune
Finetunes encoder using cosine similarity (goal: explicitly encode same label closer, different label further), reuses existing logic from `embed_dataset.py` embedding pipeline (chunking + averaging) + tweak to pass finetuned encoder instead of pretrained to produce embeddings from the finetuned encoder

In [1]:
# Install dependencies
%pip install -q transformers==5.5.0 datasets accelerate evaluate scikit-learn sentence-transformers importlib-metadata torch huggingface_hub torchao==0.16.0

import transformers
import torch

print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)

# Only use 1 GPU -- no dataparallel, disable progress bars so no more crashing even though everything's fine
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["DISABLE_PROGRESS_BARS"] = "1"

from transformers.utils import logging
logging.disable_progress_bar()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 99.9 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.
Transformers version: 5.5.0
Torch version: 2.10.0+cu128


In [2]:
# Config - edit these before running (MODEL_NAME and DATA_CSV are required)
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
DATA_CSV = '/kaggle/input/datasets/faithkalendek/synthetic-conversations-full/synthetic_conversations_full.csv'
# Output folders (each run will create subfolders under these)
FINETUNED_DIR = '/kaggle/working/runs/b8e10lr2e-3wu30'
OUT_DIR = '/kaggle/working/runs/b8e10lr2e-3wu30/embeddings'

# Training settings
TRAIN_FP16 = True        
TRAIN_BATCH_SIZE = 16  # contrastive batch size (in-batch negatives)
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5        
WARMUP_STEPS = 30

# Embedding batch size used by load_and_embed when saving embeddings
BATCH_SIZE = 32

In [3]:
# Contrastive training using SentenceTransformers (in-batch negatives via MultipleNegativesRankingLoss)
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
import os

# Read CSV and build items (text, label, num_turns) -- reuse same expansion as embedding pipeline
df = pd.read_csv(DATA_CSV, dtype=str).fillna("")
output_cols = sorted([c for c in df.columns if c.startswith('llm_output_')], key=lambda s: int(s.split('_')[-1]) if s.split('_')[-1].isdigit() else s)
rows = []
for _, row in df.iterrows():
    trait = row.get('trait_selected', row.get('trait', '0'))
    try:
        label = int(trait) if trait != '' else 0
    except Exception:
        label = 0
    outputs = []
    for col in output_cols:
        txt = row.get(col, '')
        if txt is None:
            txt = ''
        txt = str(txt).strip()
        if txt != '':
            outputs.append(txt)
    if len(outputs) == 0:
        continue
    # Don't use diff convo length samples for training -- can produce pairs of nearly identical text
    text = '\n\n'.join(outputs)
    rows.append({'text': text, 'label': label})

meta_df = pd.DataFrame(rows)

# Train / Eval split (stratify by label+num_turns when possible)
try:
    strat = meta_df['label'].astype(str) + '_' + meta_df['num_turns'].astype(str)
    train_df, val_df = train_test_split(meta_df, test_size=0.15, stratify=strat, random_state=42)
except Exception:
    train_df, val_df = train_test_split(meta_df, test_size=0.15, stratify=meta_df['label'], random_state=42)

# Build positive pairs for contrastive training: for each label, pair each example with a different example of same label (wrap-around)
from collections import defaultdict
groups = defaultdict(list)
for _, r in train_df.iterrows():
    groups[int(r['label'])].append(r['text'])

import random

train_examples = []
for lab, texts in groups.items():
    if len(texts) < 2:
        continue
    for i, a in enumerate(texts):
        others = texts[:i] + texts[i+1:]
        sampled = random.sample(others, min(3, len(others)))  # 3 positives per anchor
        for b in sampled:
            train_examples.append(InputExample(texts=[a, b]))

print('Built', len(train_examples), 'contrastive training pairs')

# Device detection (reuse simple CUDA/directml/CPU logic)
import torch, importlib
if torch.cuda.is_available():
    device = 'cuda'
else:
    try:
        td = importlib.import_module('torch_directml')
        device = td.device()
    except Exception:
        device = 'cpu'

# Initialize SentenceTransformer model and prepare dataloader
model = SentenceTransformer(MODEL_NAME, device=device)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=TRAIN_BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(model)

import torch, os
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# Fit model (writes model to FINETUNED_DIR). This trains the encoder directly with contrastive loss.
os.makedirs(FINETUNED_DIR, exist_ok=True)
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=NUM_EPOCHS, warmup_steps=WARMUP_STEPS, output_path=FINETUNED_DIR, use_amp=TRAIN_FP16)

# Save trained SentenceTransformer and expose as st_model for downstream embedding steps
print('Saved contrastively-finetuned model to', FINETUNED_DIR)
st_model = SentenceTransformer(FINETUNED_DIR, device=device)
globals()['st_model'] = st_model
print('st_model set; ready to embed with the finetuned encoder')

# --- sanity check: are embeddings separating at all? ---
sample_texts = train_df['text'].tolist()[:200]
sample_labels = train_df['label'].tolist()[:200]

emb = st_model.encode(sample_texts)

from sklearn.metrics.pairwise import cosine_similarity
same, diff = [], []

for i in range(len(sample_texts)):
    for j in range(i+1, len(sample_texts)):
        sim = cosine_similarity([emb[i]], [emb[j]])[0][0]
        if sample_labels[i] == sample_labels[j]:
            same.append(sim)
        else:
            diff.append(sim)

print("avg same-class similarity:", np.mean(same))
print("avg diff-class similarity:", np.mean(diff))

Built 1449 contrastive training pairs


MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,2.031522


Saved contrastively-finetuned model to /kaggle/working/runs/b8e10lr2e-3wu30
st_model set; ready to embed with the finetuned encoder
avg same-class similarity: 0.774518
avg diff-class similarity: 0.14752786


In [4]:
# Load finetuned SentenceTransformer (preferred) or fall back to base model for embedding
import importlib
import torch
import os
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

# Detect best available device: CUDA -> DirectML -> CPU
if torch.cuda.is_available():
    device = "cuda"
else:
    try:
        td = importlib.import_module("torch_directml")
        device = td.device()
    except Exception:
        device = "cpu"

# Try to load a SentenceTransformer saved in FINETUNED_DIR (contrastive finetune output)
st_model = None
try:
    if os.path.isdir(FINETUNED_DIR):
        st_model = SentenceTransformer(FINETUNED_DIR, device=device)
        print('Loaded finetuned SentenceTransformer from', FINETUNED_DIR)
    else:
        raise FileNotFoundError
except Exception:
    # Fallback: load base SentenceTransformer for tokenization/encode
    print('Finetuned SentenceTransformer not found in FINETUNED_DIR; loading base model instead')
    st_model = SentenceTransformer(MODEL_NAME, device=device)

# Expose tokenizer and model objects used by embedding logic
try:
    hf_tokenizer = st_model._first_module().tokenizer
except Exception:
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# We do not require a HF AutoModel when using st_model.encode; keep a lightweight fallback
try:
    hf_model = st_model._first_module()
except Exception:
    hf_model = AutoModel.from_pretrained(MODEL_NAME)
    hf_model.to(device)
    hf_model.eval()

globals()['st_model'] = st_model
globals()['hf_model'] = hf_model
globals()['hf_tokenizer'] = hf_tokenizer
print('st_model, hf_model, hf_tokenizer set; ready to embed using the finetuned encoder (if available)')

Loaded finetuned SentenceTransformer from /kaggle/working/runs/b8e10lr2e-3wu30
st_model, hf_model, hf_tokenizer set; ready to embed using the finetuned encoder (if available)


In [5]:
# embed_dataset.py logic
import numpy as np, os
from transformers import AutoTokenizer, AutoModel
import torch

# HF tokenizer/model lazily used only to avoid inference-mode tensor issues
# hf_tokenizer and hf_model set in cell above
# If not set, fall back to loading the base model
# Use `# noqa` to silence style/lint checks when a line is intentionally atypical
try:
    hf_tokenizer  # noqa
except NameError:
    print("HF tokenizer not found, loading base model tokenizer...")
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

try:
    hf_model  # noqa
except NameError:
    print("HF model not found, loading base model...")
    hf_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    hf_model.eval()

# Reuse st_model (SentenceTransformer) for tokenization and max seq length helpers
model = st_model

def chunk_and_aggregate(text) :
    # Use HuggingFace tokenizer to split by tokens -- chunks should overlap a little bit, but not too much (maybe 30 token overlap?)
    tokens = model.tokenize(text) # returns dict, NOT list of tokens
    token_list = tokens['input_ids'][0].tolist() # get list of token ids from batch of 1
    
    # Max length of tokens for this model
    max_length = model.get_max_seq_length()
    
    chunks = [token_list[i:i+max_length] for i in range(0, len(token_list), max_length - 30)]  # 30 token overlap

    # Decode token-id chunks back to text before calling `model.encode` to avoid
    # creating inference-mode tensors from raw id lists (which can trigger
    # "Cannot set version_counter for inference tensor" errors with some models).
    tokenizer = None
    try:
        tokenizer = model._first_module().tokenizer
    except Exception:
        tokenizer = None

    if tokenizer is not None:
        chunk_texts = [tokenizer.decode(c, skip_special_tokens=True) for c in chunks]
    else:
        chunk_texts = [" ".join(map(str, c)) for c in chunks]

    # If a finetuned SentenceTransformer is available (st_model), prefer its encode method
    # since it encapsulates tokenization + pooling and respects the finetuned encoder.
    if 'st_model' in globals() and st_model is not None:
        chunk_embeddings = st_model.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=False)
        aggregated_embedding = np.mean(chunk_embeddings, axis=0)
        return aggregated_embedding

    # Fallback: Get embeddings for each chunk and average them using HF AutoModel + tokenizer.
    global hf_tokenizer, hf_model
    if hf_tokenizer is None or hf_model is None:
        hf_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
        hf_model = AutoModel.from_pretrained('sentence-transformers/all-mpnet-base-v2')
        hf_model.to(device)

    tokens = hf_tokenizer(chunk_texts, padding=True, truncation=True, return_tensors='pt')
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.no_grad():
        out = hf_model(**tokens, return_dict=True)
    last_hidden = out.last_hidden_state  # (batch, seq_len, dim)
    mask = tokens['attention_mask'].unsqueeze(-1).to(last_hidden.dtype)
    pooled = (last_hidden * mask).sum(1) / mask.sum(1)
    chunk_embeddings = pooled.cpu().numpy()
    aggregated_embedding = chunk_embeddings.mean(axis=0)
    
    return aggregated_embedding


def embed_batch(batch):
    embeddings = []
    labels = []
    metadata = [] # trait and num_turns -- for doing stratified split later in the observer logic

    # DataLoader collates batch into a tuple (texts_list, labels_list, ks_list) instead of list of tuples
    texts, labs, ks = batch
    for response, label, k in zip(texts, labs, ks):
        label_val = int(label.item()) if isinstance(label, torch.Tensor) else int(label)
        embedding = chunk_and_aggregate(response)
        embeddings.append(embedding)
        labels.append(label_val)
        metadata.append({'trait': label_val, 'num_turns': int(k)})
    
    # Stack into single ndarray first before converting to tensor, since list of np arrays --> tensor is super slow apparantly
    batch_embeddings = np.vstack(embeddings)
    batch_labels = np.array(labels, dtype=int)
    return torch.from_numpy(batch_embeddings), torch.from_numpy(batch_labels), metadata


# Embed all responses in the dataset and save to .npy files
import numpy as np
import os
def embed_dataset(dataloader, out_dir='embeddings') :
    os.makedirs(out_dir, exist_ok=True)
    
    all_embeddings = []
    all_labels = []
    metadata = []
    
    for batch in dataloader:
        batch_embeddings, batch_labels, batch_metadata = embed_batch(batch)
        all_embeddings.append(batch_embeddings)
        all_labels.append(batch_labels)
        metadata.append(batch_metadata)
        
    all_embeddings = torch.cat(all_embeddings).numpy()
    all_labels = torch.cat(all_labels).numpy()
    metadata = [item for sublist in metadata for item in sublist]
    
    np.save(os.path.join(out_dir, 'embeddings.npy'), all_embeddings)
    np.save(os.path.join(out_dir, 'labels.npy'), all_labels)
    
    # Save metadata as JSON
    import json
    
    # Metadata is a list of tensors, convert to list of dicts first
    metadata = [{'trait': int(m['trait']), 'num_turns': int(m['num_turns'])} for m in metadata]
    
    with open(os.path.join(out_dir, 'metadata.json'), 'w') as f:
        json.dump(metadata, f)
    
    
# Load responses from csv, concatenate responses for each convo length, and embed
import pandas as pd
def load_and_embed(csv_path, out_dir='embeddings', batch_size=32):
   
    df = pd.read_csv(csv_path, dtype=str).fillna("")

    # Get llm output columns (e.g. llm_output_0 .. llm_output_4)
    output_cols = sorted([c for c in df.columns if c.startswith('llm_output_')],
                         key=lambda s: int(s.split('_')[-1]) if s.split('_')[-1].isdigit() else s)

    items = []  # list of (concat_text, label)

    for _, row in df.iterrows():
        # Try trait_selected, fallback to trait or '0'
        trait = row.get('trait_selected', row.get('trait', '0'))
        try:
            label = int(trait) if trait != '' else 0
        except Exception:
            label = 0

        outputs = []
        for col in output_cols:
            txt = row.get(col, "")
            if txt is None:
                txt = ""
            txt = str(txt).strip()
            if txt != "":
                outputs.append(txt)

        if len(outputs) == 0:
            continue

        # create concatenated texts for each convo length, include num_turns in each item for stratified splitting later
        for k in range(1, len(outputs) + 1):
            concat = "\n\n".join(outputs[:k])
            items.append((concat, label, k))  # (text, label, num_turns)

    # Initialize dataloader and embed dataset
    dataloader = torch.utils.data.DataLoader(items, batch_size=batch_size, shuffle=False)
    embed_dataset(dataloader, out_dir)

In [6]:
load_and_embed(DATA_CSV, out_dir=OUT_DIR, batch_size=BATCH_SIZE)
print('Saved embeddings to', OUT_DIR)

Saved embeddings to /kaggle/working/runs/b8e10lr2e-3wu30/embeddings


In [7]:
# Evaluation & diagnostics for embeddings
# Loads embeddings from OUT_DIR and computes:
# - duplicate text check (reconstructs texts from DATA_CSV)
# - embedding norms stats
# - silhouette (cosine)
# - kNN recall@1/3/5 (cosine)
# - silhouette after removing top PCs (1..3)
# - logistic probe accuracy (simple train/test split)
# Saves metrics to OUT_DIR/metrics_contrastive.json

import os, json
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

emb_path = os.path.join(OUT_DIR, 'embeddings.npy')
labels_path = os.path.join(OUT_DIR, 'labels.npy')
metrics_out = os.path.join(OUT_DIR, 'metrics_contrastive.json')

if not os.path.exists(emb_path) or not os.path.exists(labels_path):
    raise FileNotFoundError(f"Embeddings or labels not found in {OUT_DIR}. Run embedding cell first.")

E = np.load(emb_path)
labels = np.load(labels_path)
print('Loaded embeddings:', E.shape, 'labels:', labels.shape)

results = {}

# Norms
norms = np.linalg.norm(E, axis=1)
results['norm_mean'] = float(np.mean(norms))
results['norm_std'] = float(np.std(norms))
results['norm_min'] = float(np.min(norms))
results['norm_max'] = float(np.max(norms))
results['norm_25q'] = float(np.quantile(norms, 0.25))
results['norm_50q'] = float(np.quantile(norms, 0.5))
results['norm_75q'] = float(np.quantile(norms, 0.75))
print('Norms mean/std:', results['norm_mean'], results['norm_std'])

# L2-normalize for cosine-based metrics
E_norm = E / np.linalg.norm(E, axis=1, keepdims=True)

# Silhouette (cosine)
try:
    sil = silhouette_score(E_norm, labels, metric='cosine')
except Exception as e:
    sil = None
    print('Silhouette score failed:', e)
results['silhouette_cosine'] = float(sil) if sil is not None else None
print('Silhouette (cosine):', sil)

# kNN recall@k (cosine)
nbr = NearestNeighbors(n_neighbors=6, metric='cosine').fit(E_norm)
dist, idx = nbr.kneighbors(E_norm)
# idx[:,0] is self

def recall_at_k(k):
    # k: 1 means nearest neighbor excluding self -> idx[:,1]
    matches = []
    for i in range(len(labels)):
        neighbor = idx[i, k]  # k-th neighbor (0=self, 1=nn)
        matches.append(int(labels[i] == labels[neighbor]))
    return float(np.mean(matches))

for k in [1,2,3,5]:
    # guard k index within neighbors (k refers to neighbor rank)
    kk = min(k, idx.shape[1]-1)
    results[f'recall_at_{k}'] = recall_at_k(kk)
    print(f'recall@{k}:', results[f'recall_at_{k}'])

# Silhouette after removing top PCs
def remove_top_pcs(X, n_remove=1):
    pca = PCA(n_components=min(50, X.shape[1]))
    Z = pca.fit_transform(X)
    Z[:, :n_remove] = 0
    Xr = pca.inverse_transform(Z)
    return Xr

for r in [1,2,3]:
    try:
        Er = remove_top_pcs(E, r)
        Ern = Er / np.linalg.norm(Er, axis=1, keepdims=True)
        s = silhouette_score(Ern, labels, metric='cosine')
    except Exception as e:
        s = None
        print('PC removal silhouette failed for r=', r, e)
    results[f'silhouette_remove_top_{r}_pcs'] = float(s) if s is not None else None
    print(f'silhouette after removing top {r} PCs:', s)

# Logistic probe (simple) on normalized embeddings
X_train, X_test, y_train, y_test = train_test_split(E_norm, labels, test_size=0.2, stratify=labels, random_state=42)
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_s, y_train)
y_pred = clf.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
results['logistic_probe_accuracy'] = float(acc)
print('Logistic probe accuracy:', acc)

# Duplicate texts / label leakage check: reconstruct texts in same order as embed_dataset
from collections import defaultdict

def build_items_from_csv(csv_path):
    df = pd.read_csv(csv_path, dtype=str).fillna("")
    output_cols = sorted([c for c in df.columns if c.startswith('llm_output_')], key=lambda s: int(s.split('_')[-1]) if s.split('_')[-1].isdigit() else s)
    items = []
    for _, row in df.iterrows():
        trait = row.get('trait_selected', row.get('trait', '0'))
        try:
            label = int(trait) if trait != '' else 0
        except Exception:
            label = 0
        outputs = []
        for col in output_cols:
            txt = row.get(col, "")
            if txt is None:
                txt = ""
            txt = str(txt).strip()
            if txt != "":
                outputs.append(txt)
        if len(outputs) == 0:
            continue
        for k in range(1, len(outputs) + 1):
            concat = "\n\n".join(outputs[:k])
            items.append((concat, label, k))
    return items

items = build_items_from_csv(DATA_CSV)
texts = [t for t,_,_ in items]
labels_from_items = [l for _,l,_ in items]
print('Reconstructed', len(texts), 'texts from CSV')

# Find exact duplicates mapping to multiple labels
d = defaultdict(list)
for i,t in enumerate(texts):
    key = t.strip()
    d[key].append((i, labels_from_items[i]))

dups = [(k,v) for k,v in d.items() if len(v) > 1]
results['n_duplicate_texts'] = len(dups)
print('Duplicate text groups found:', results['n_duplicate_texts'])
# show up to 5 examples
for k,v in dups[:5]:
    print('--- DUPLICATE SNIPPET ---')
    print(k[:200])
    print('items:', v[:10])

# Save metrics
os.makedirs(OUT_DIR, exist_ok=True)
with open(metrics_out, 'w') as f:
    json.dump(results, f, indent=2)
print('Saved metrics to', metrics_out)


Loaded embeddings: (2845, 768) labels: (2845,)
Norms mean/std: 1.0 1.9515914218004582e-08
Silhouette (cosine): -0.17305185
recall@1: 0.26362038664323373
recall@2: 0.24253075571177504
recall@3: 0.21616871704745166
recall@5: 0.17926186291739896
silhouette after removing top 1 PCs: -0.20868814
silhouette after removing top 2 PCs: -0.27459314
silhouette after removing top 3 PCs: -0.2565283
Logistic probe accuracy: 0.3866432337434095
Reconstructed 2845 texts from CSV
Duplicate text groups found: 0
Saved metrics to /kaggle/working/runs/b8e10lr2e-3wu30/embeddings/metrics_contrastive.json
